In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

In [2]:
transactions = pd.read_csv(
    '../data/bronze/transactions_history_final.csv',
    low_memory=False
)

outlet_master = pd.read_csv('../data/bronze/outlet_master.csv')

outlet_coordinates = pd.read_csv('../data/bronze/outlet_coordinates.csv')

seasonality = pd.read_csv(
    '../data/bronze/distributor_seasonality_details.csv'
)

holidays = pd.read_csv('../data/bronze/holiday_list.csv')

REJECT LOGGER FUNCTION

In [3]:
def create_rejected_store(df, reason):

    rejected = df.copy()

    rejected['rejection_reason'] = reason

    rejected['rejected_timestamp'] = datetime.now()

    return rejected

REUSABLE DATA QUALITY CHECKS

CHECK 1 — NULL CHECK

In [4]:
def null_check(df, required_columns):

    mask = df[required_columns].isnull().any(axis=1)

    failed = df[mask]

    passed = df[~mask]

    return passed, failed

RUN NULL CHECK

In [6]:
required_columns = [
    'Outlet_ID',
    'Distributor_ID',
    'SKU_ID',
    'Volume_Liters',
    'Total_Bill_Value'
]

transactions_clean, transactions_nulls = null_check(
    transactions,
    required_columns
)

transactions_nulls = create_rejected_store(
    transactions_nulls,
    'NULL_CHECK_FAILED'
)

print(transactions_nulls.shape)

(0, 9)


DUPLICATE CHECK

In [7]:
def duplicate_check(df, subset_cols):

    mask = df.duplicated(
        subset=subset_cols,
        keep='first'
    )

    failed = df[mask]

    passed = df[~mask]

    return passed, failed

In [8]:
dup_cols = [
    'Outlet_ID',
    'Year',
    'Month',
    'Distributor_ID',
    'SKU_ID',
    'Volume_Liters',
    'Total_Bill_Value'
]

transactions_clean, duplicate_rows = duplicate_check(
    transactions_clean,
    dup_cols
)

duplicate_rows = create_rejected_store(
    duplicate_rows,
    'DUPLICATE_RECORD'
)

print(duplicate_rows.shape)

(0, 9)


VALUE RANGE CHECKS

In [9]:
def range_check(df, column, min_val=None, max_val=None):

    mask = pd.Series(False, index=df.index)

    if min_val is not None:
        mask |= df[column] < min_val

    if max_val is not None:
        mask |= df[column] > max_val

    failed = df[mask]

    passed = df[~mask]

    return passed, failed

CHECK NEGATIVE VOLUMES

In [11]:
transactions_clean, negative_volume = range_check(
    transactions_clean,
    'Volume_Liters',
    min_val=0
)

negative_volume = create_rejected_store(
    negative_volume,
    'NEGATIVE_VOLUME'
)

print(negative_volume.shape)

(4753, 9)


CHECK NEGATIVE BILL VALUES

In [12]:
transactions_clean, negative_bill = range_check(
    transactions_clean,
    'Total_Bill_Value',
    min_val=0
)

negative_bill = create_rejected_store(
    negative_bill,
    'NEGATIVE_BILL_VALUE'
)

print(negative_bill.shape)

(0, 9)


ZERO VOLUME GHOSTS

In [13]:
zero_volume = transactions_clean[
    transactions_clean['Volume_Liters'] == 0
]

zero_volume = create_rejected_store(
    zero_volume,
    'ZERO_VOLUME_GHOST'
)

transactions_clean = transactions_clean[
    transactions_clean['Volume_Liters'] != 0
]

print(zero_volume.shape)

(100, 9)


REFERENTIAL INTEGRITY CHECKS

In [14]:
def referential_integrity_check(
    df,
    column,
    reference_values
):

    mask = ~df[column].isin(reference_values)

    failed = df[mask]

    passed = df[~mask]

    return passed, failed

CHECK OUTLET IDS

In [16]:
valid_outlets = outlet_master['Outlet_ID'].unique()

transactions_clean, invalid_outlets = referential_integrity_check(
    transactions_clean,
    'Outlet_ID',
    valid_outlets
)

invalid_outlets = create_rejected_store(
    invalid_outlets,
    'INVALID_OUTLET_ID'
)

print(invalid_outlets.shape)

(0, 9)


CHECK DISTRIBUTOR IDS

In [17]:
valid_distributors = seasonality['Distributor_ID'].unique()

transactions_clean, invalid_distributors = referential_integrity_check(
    transactions_clean,
    'Distributor_ID',
    valid_distributors
)

invalid_distributors = create_rejected_store(
    invalid_distributors,
    'INVALID_DISTRIBUTOR_ID'
)

print(invalid_distributors.shape)

(0, 9)


MONTH RANGE CHECK

In [18]:
invalid_months = transactions_clean[
    ~transactions_clean['Month'].between(1, 12)
]

invalid_months = create_rejected_store(
    invalid_months,
    'INVALID_MONTH'
)

transactions_clean = transactions_clean[
    transactions_clean['Month'].between(1, 12)
]

YEAR CHECK

In [19]:
invalid_years = transactions_clean[
    ~transactions_clean['Year'].between(2023, 2025)
]

invalid_years = create_rejected_store(
    invalid_years,
    'INVALID_YEAR'
)

transactions_clean = transactions_clean[
    transactions_clean['Year'].between(2023, 2025)
]

REVENUE PER LITER FORENSICS

In [20]:
transactions_clean['Revenue_Per_Liter'] = (
    transactions_clean['Total_Bill_Value'] /
    transactions_clean['Volume_Liters']
)

In [21]:
Q1 = transactions_clean['Revenue_Per_Liter'].quantile(0.25)
Q3 = transactions_clean['Revenue_Per_Liter'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

pricing_anomalies = transactions_clean[
    (
        transactions_clean['Revenue_Per_Liter'] < lower
    ) |
    (
        transactions_clean['Revenue_Per_Liter'] > upper
    )
]

pricing_anomalies = create_rejected_store(
    pricing_anomalies,
    'REVENUE_PER_LITER_ANOMALY'
)

print(pricing_anomalies.shape)

(236818, 10)


COORDINATE VALIDATION

In [22]:
invalid_coords = outlet_coordinates[
    (
        (outlet_coordinates['Latitude'] < 5.5) |
        (outlet_coordinates['Latitude'] > 10)
    ) |
    (
        (outlet_coordinates['Longitude'] < 79) |
        (outlet_coordinates['Longitude'] > 82)
    )
]

invalid_coords = create_rejected_store(
    invalid_coords,
    'INVALID_COORDINATES'
)

print(invalid_coords.shape)

(240, 5)


MASTER DATA DECAY DETECTION

In [24]:
outlet_behavior = (
    transactions_clean
    .groupby('Outlet_ID')['Volume_Liters']
    .sum()
    .reset_index()
)

In [25]:
outlet_behavior = outlet_behavior.merge(
    outlet_master,
    on='Outlet_ID',
    how='left'
)

IDENTIFY SUSPICIOUSLY HIGH “SMALL” SHOPS

In [42]:
suspicious_small = outlet_behavior[
    (
        outlet_behavior['Outlet_Size'] == 'Small'
    ) &
    (
        outlet_behavior['Volume_Liters'] >
        outlet_behavior['Volume_Liters'].quantile(0.95)
    )
]

print(suspicious_small.shape)

(2, 5)


In [43]:
suspicious_small.head(20)

,Outlet_ID,Volume_Liters,Outlet_Size,Cooler_Count,Outlet_Type
104,OUT_00105,20315.269034,Small,1,Bakery
380,OUT_00381,20130.348906,Small,1,Grocery


In [57]:
duplicate_rows.head(20)

,Outlet_ID,Year,Month,Distributor_ID,SKU_ID,Volume_Liters,Total_Bill_Value,rejection_reason,rejected_timestamp


In [55]:
negative_volume.head(20)

,Outlet_ID,Year,Month,Distributor_ID,SKU_ID,Volume_Liters,Total_Bill_Value,rejection_reason,rejected_timestamp
234,OUT_19987,2024,12,DIST_S_02,SKU_07,-20.975223,-13633.905373,NEGATIVE_VOLUME,2026-05-16 17:16:02.842703
1097,OUT_14664,2025,4,DIST_NW_01,SKU_02,-142.662371,-36141.107438,NEGATIVE_VOLUME,2026-05-16 17:16:02.842703
1209,OUT_13596,2023,6,DIST_NW_02,SKU_05,-72.336385,-6510.268246,NEGATIVE_VOLUME,2026-05-16 17:16:02.842703
2278,OUT_10231,2025,2,DIST_C_03,SKU_05,-64.165540,-5774.896514,NEGATIVE_VOLUME,2026-05-16 17:16:02.842703
3621,OUT_17901,2024,7,DIST_S_01,SKU_08,-6.006375,-2642.768002,NEGATIVE_VOLUME,2026-05-16 17:16:02.842703
5539,OUT_16990,2024,2,DIST_NW_01,SKU_04,-48.167168,-15654.295884,NEGATIVE_VOLUME,2026-05-16 17:16:02.842703
6358,OUT_14981,2023,8,DIST_NW_01,SKU_10,-19.292824,-7123.481591,NEGATIVE_VOLUME,2026-05-16 17:16:02.842703
6536,OUT_17208,2023,4,DIST_S_02,SKU_05,-90.313231,-8128.192930,NEGATIVE_VOLUME,2026-05-16 17:16:02.842703
6539,OUT_13324,2024,6,DIST_NW_02,SKU_04,-9.309313,-3025.536013,NEGATIVE_VOLUME,2026-05-16 17:16:02.842703
7641,OUT_02956,2023,7,DIST_W_02,SKU_08,-33.402439,-14697.091953,NEGATIVE_VOLUME,2026-05-16 17:16:02.842703


In [58]:
transactions['SKU_ID'].value_counts(normalize=True)

SKU_ID
SKU_01    0.121042
SKU_07    0.097900
SKU_10    0.097769
SKU_04    0.097731
SKU_06    0.097655
SKU_03    0.097629
SKU_02    0.097612
SKU_05    0.097571
SKU_08    0.097555
SKU_09    0.097537
Name: proportion, dtype: float64

In [59]:
transactions.groupby('Distributor_ID')['Volume_Liters'].describe()

,count,mean,std,min,25%,50%,75%,max
Distributor_ID,,,,,,,,
DIST_C_01,164284.0,55.393308,99.936151,-620.513597,10.660051,24.189182,56.780159,1093.666849
DIST_C_02,152354.0,53.905670,97.597772,-662.257106,10.489717,23.728192,55.663821,1090.630676
DIST_C_03,153129.0,58.069333,106.769097,-956.440795,10.561545,24.558315,58.012113,1142.490690
DIST_NW_01,239818.0,54.887272,98.832641,-906.666415,10.526210,24.139342,56.679592,1221.573307
DIST_NW_02,236101.0,53.529305,96.054326,-842.864780,10.500079,23.796472,55.804731,1096.679597
DIST_S_01,167376.0,49.897377,93.281047,-846.356312,9.197350,20.884726,49.097548,1120.710503
DIST_S_02,168795.0,51.785265,99.077813,-761.713987,9.562079,21.883490,51.641827,9438.577616
DIST_W_01,366300.0,51.557925,92.584059,-807.724020,10.066630,22.875376,53.722765,1135.441637
DIST_W_02,365220.0,50.144623,89.299878,-932.141798,10.132375,22.532236,52.032233,1233.045264


In [60]:
invalid_coords.head(20)

,Outlet_ID,Latitude,Longitude,rejection_reason,rejected_timestamp
29,OUT_00030,79.923311,7.188307,INVALID_COORDINATES,2026-05-16 17:24:27.627310
89,OUT_00090,0.000000,0.000000,INVALID_COORDINATES,2026-05-16 17:24:27.627310
170,OUT_00171,79.999061,6.706547,INVALID_COORDINATES,2026-05-16 17:24:27.627310
251,OUT_00252,80.032979,7.134312,INVALID_COORDINATES,2026-05-16 17:24:27.627310
456,OUT_00457,80.008335,6.712571,INVALID_COORDINATES,2026-05-16 17:24:27.627310
510,OUT_00511,80.019915,6.866704,INVALID_COORDINATES,2026-05-16 17:24:27.627310
560,OUT_00561,79.854092,6.973430,INVALID_COORDINATES,2026-05-16 17:24:27.627310
581,OUT_00582,79.888708,6.827655,INVALID_COORDINATES,2026-05-16 17:24:27.627310
599,OUT_00600,79.968622,7.063528,INVALID_COORDINATES,2026-05-16 17:24:27.627310
677,OUT_00678,79.979931,6.792686,INVALID_COORDINATES,2026-05-16 17:24:27.627310


SAVE REJECTED RECORDS

In [28]:
rejected_all = pd.concat([

    transactions_nulls,
    duplicate_rows,
    negative_volume,
    negative_bill,
    zero_volume,
    invalid_outlets,
    invalid_distributors,
    invalid_months,
    invalid_years,
    pricing_anomalies

], ignore_index=True)

In [29]:
rejected_all.to_csv(
    '../data/rejected/rejected_records.csv',
    index=False
)

SAVE SILVER DATASETS

In [30]:
transactions_clean.to_csv(
    '../data/silver/transactions_clean.csv',
    index=False
)

In [31]:
clean_coords = outlet_coordinates.drop(
    invalid_coords.index
)

clean_coords.to_csv(
    '../data/silver/outlet_coordinates_clean.csv',
    index=False
)

In [32]:
outlet_master.to_csv(
    '../data/silver/outlet_master_clean.csv',
    index=False
)

seasonality.to_csv(
    '../data/silver/seasonality_clean.csv',
    index=False
)

holidays.to_csv(
    '../data/silver/holiday_clean.csv',
    index=False
)

CREATING DATA QUALITY REPORT

In [33]:
dq_summary = pd.DataFrame({

    'Check': [
        'Null Check',
        'Duplicate Check',
        'Negative Volume',
        'Negative Bill',
        'Zero Volume',
        'Invalid Outlet',
        'Invalid Distributor',
        'Invalid Month',
        'Invalid Year',
        'Pricing Anomalies'
    ],

    'Rejected_Count': [
        len(transactions_nulls),
        len(duplicate_rows),
        len(negative_volume),
        len(negative_bill),
        len(zero_volume),
        len(invalid_outlets),
        len(invalid_distributors),
        len(invalid_months),
        len(invalid_years),
        len(pricing_anomalies)
    ]
})

In [34]:
dq_summary.to_csv(
    '../data/reports/data_quality_summary.csv',
    index=False
)

dq_summary

,Check,Rejected_Count
0,Null Check,0
1,Duplicate Check,0
2,Negative Volume,4753
3,Negative Bill,0
4,Zero Volume,100
5,Invalid Outlet,0
6,Invalid Distributor,0
7,Invalid Month,0
8,Invalid Year,0
9,Pricing Anomalies,236818
